In [6]:
# --- Connect to the cohort database (read-only) ---
import sys, pathlib, re
_p = pathlib.Path.cwd()
while _p != _p.parent and not (_p / "db" / "dataraflow_db.py").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))
from db.dataraflow_db import run_query, get_engine, get_connection

import pandas as pd
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

_BLOCKED = re.compile(
    r"\b(INSERT|UPDATE|DELETE|MERGE|DROP|ALTER|CREATE|TRUNCATE|EXEC|EXECUTE|GRANT|REVOKE|DENY|INTO|BACKUP|RESTORE)\b",
    re.IGNORECASE,
)

def q(sql: str) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.

    String literals and comments are stripped before the safety check, so a value
    like 'Delete me' inside quotes is fine.
    """
    scrubbed = re.sub(r"'([^']|'')*'", "''", sql)           # string literals
    scrubbed = re.sub(r"--[^\n]*", "", scrubbed)              # -- comments
    scrubbed = re.sub(r"/\*.*?\*/", "", scrubbed, flags=re.S) # /* comments */
    hit = _BLOCKED.search(scrubbed)
    if hit:
        raise PermissionError(
            f"'{hit.group(0).upper()}' is not allowed: this notebook is read-only. "
            "Shape your data inside the SELECT, or in pandas."
        )
    return run_query(sql)

q("SELECT DB_NAME() AS database_name, CURRENT_USER AS logged_in_as, SYSDATETIME() AS server_time")



    




,database_name,logged_in_as,server_time
0,assessmentDB,intern_readonly,2026-10-14 09:15:16.625807


In [5]:
try:
    q("DELETE FROM week06.london_2014")
except PermissionError as e:
    print("Blocked as expected ->", e)

Blocked as expected -> 'DELETE' is not allowed: this notebook is read-only. Shape your data inside the SELECT, or in pandas.


In [7]:
q("""
    SELECT TABLE_SCHEMA + '.' + TABLE_NAME AS table_name, COUNT(*) AS n_columns
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA IN ('week16', 'week17')
    GROUP BY TABLE_SCHEMA, TABLE_NAME
    ORDER BY n_columns DESC
""")

,table_name,n_columns
0,week16.customer_churn_prediction,19
1,week17.loan_approval_prediction,16
2,week16.assignment3_credit_risk,10
3,week17.assignment2_medical_diagnosis,10
4,week16.assignment2_marketing_campaign,9
5,week16.task3_classification_data,9
6,week16.assignment1_house_prices,8
7,week17.assignment1_product_recommendation,8
8,week17.assignment3_classifier_comparison,7
9,week17.task3_naive_bayes_data,7


In [8]:
q("""
    SELECT COUNT(*)                       AS n_rows,
           COUNT(DISTINCT Contract_Type)  AS n_contract_types,
           SUM(Churn)                     AS n_churned,
           MIN(Monthly_Charges)           AS min_monthly,
           MAX(Monthly_Charges)           AS max_monthly
    FROM week16.customer_churn_prediction
""")


,n_rows,n_contract_types,n_churned,min_monthly,max_monthly
0,500,3,228,20.03,119.20


In [9]:
q("""
    SELECT TOP 10
           country,
           [NY.GDP.MKTP.CD] / 1e9 AS gdp_billion_usd
    FROM   week08.wb_gdp_2013
    WHERE  [NY.GDP.MKTP.CD] IS NOT NULL
    ORDER BY gdp_billion_usd DESC
""")

,country,gdp_billion_usd
0,World,"76,236.80"
1,High income,"52,256.19"
2,OECD members,"48,009.76"
3,High income: OECD,"45,927.31"
4,Low & middle income,"24,011.49"
5,Middle income,"23,637.12"
6,Europe & Central Asia (all income levels),"22,949.90"
7,East Asia & Pacific (all income levels),"20,846.73"
8,North America,"18,612.59"
9,Upper middle income,"18,155.01"


In [10]:
q("""
    SELECT DISTINCT [Trade Flow], [Commodity Code], Commodity
    FROM week09.comtrade_milk_uk_monthly_14
    ORDER BY [Trade Flow], [Commodity Code]
""")

,Trade Flow,Commodity Code,Commodity
0,Exports,0401,"Milk and cream, neither concentrated nor sweet..."
1,Exports,0402,"Milk and cream, concentrated or sweetened"
2,Imports,0401,"Milk and cream, neither concentrated nor sweet..."
3,Imports,0402,"Milk and cream, concentrated or sweetened"


In [11]:
q("""
    SELECT TOP 5 WITH TIES GMT, [Max TemperatureC]
    FROM week06.london_2014
    ORDER BY [Max TemperatureC] DESC
""")


,GMT,Max TemperatureC
0,2014-7-18,30
1,2014-7-23,29
2,2014-7-24,29
3,2014-7-26,29
4,2014-7-17,29


In [12]:
q("""
    SELECT country, [SP.POP.TOTL] AS population
    FROM week08.wb_pop_2013
    WHERE [SP.POP.TOTL] IS NOT NULL
    ORDER BY population DESC
    OFFSET 10 ROWS FETCH NEXT 5 ROWS ONLY      -- ranks 11 to 15
""")


,country,population
0,India,"1,279,498,874.00"
1,OECD members,"1,263,178,643.00"
2,High income: OECD,"1,064,428,332.00"
3,Sub-Saharan Africa (all income levels),"948,287,652.00"
4,Sub-Saharan Africa (developing only),"947,400,670.00"


In [16]:
q("""
    SELECT TOP 10 country, ROUND([SP.POP.TOTL] / 1000.0, 1) AS pop_thousands
    FROM week08.wb_pop_2013
    WHERE [SP.POP.TOTL] IS NOT NULL
    ORDER BY [SP.POP.TOTL] ASC
""")


,country,pop_thousands
0,Tuvalu,9.90
1,Palau,20.90
2,St. Martin (French part),31.30
3,San Marino,31.40
4,Turks and Caicos Islands,33.10
5,Sint Maarten (Dutch part),36.60
6,Liechtenstein,37.00
7,Monaco,37.50
8,Faeroe Islands,48.30
9,Marshall Islands,52.80


In [17]:
q("""
    SELECT Application_ID, Property_Area, Education, Applicant_Income, Loan_Amount, Loan_Approved
    FROM week17.loan_approval_prediction
    WHERE Property_Area IN ('Urban', 'Semiurban')
      AND Applicant_Income BETWEEN 5000 AND 10000      -- inclusive on both ends
      AND NOT Education = 'Graduate'
    ORDER BY Applicant_Income
""")

,Application_ID,Property_Area,Education,Applicant_Income,Loan_Amount,Loan_Approved


In [18]:
q("""
    SELECT
        (SELECT COUNT(*) FROM week06.london_2014)                            AS all_days,
        (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] =  'Rain')     AS rain_only,
        (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] <> 'Rain')     AS not_rain_naive,
        (SELECT COUNT(*) FROM week06.london_2014 WHERE [ Events] IS NULL)       AS no_event_recorded,
        (SELECT COUNT(*) FROM week06.london_2014
          WHERE [ Events] <> 'Rain' OR [ Events] IS NULL)                          AS not_rain_correct
""")

,all_days,rain_only,not_rain_naive,no_event_recorded,not_rain_correct
0,365,202,49,114,163


In [19]:
q("""
    SELECT TOP 10
        GMT,
        COALESCE([ Events], 'No event')                        AS events_clean,
        Precipitationmm,
        [Max TemperatureC] - [Min TemperatureC]             AS temp_range,
        -- the ratio would crash on a day with zero range; NULLIF turns that into NULL instead
        Precipitationmm / NULLIF([Max TemperatureC] - [Min TemperatureC], 0) AS rain_per_degree_range
    FROM week06.london_2014
    ORDER BY Precipitationmm DESC
""")

,GMT,events_clean,Precipitationmm,temp_range,rain_per_degree_range
0,2014-8-8,Rain,21.08,7,3.01
1,2014-8-11,Rain-Thunderstorm,19.05,9,2.12
2,2014-7-28,Rain-Thunderstorm,17.02,9,1.89
3,2014-8-25,Rain,17.02,4,4.25
4,2014-1-29,Rain,13.97,2,6.99
5,2014-6-28,Rain,13.97,6,2.33
6,2014-11-23,Rain,13.97,5,2.79
7,2014-11-2,Rain,11.94,6,1.99
8,2014-11-4,Fog-Rain-Snow-Thunderstorm,11.94,10,1.19
9,2014-1-1,Rain,9.91,5,1.98


In [20]:
q("""
    SELECT COUNT(*) AS rows_with_missing
    FROM week17.loan_approval_prediction
    WHERE Gender IS NULL OR Married IS NULL OR Self_Employed IS NULL
""")

,rows_with_missing
0,0


In [21]:
q("""
    SELECT TOP 5
        GMT                                  AS raw_value,
        TRY_CONVERT(date, GMT)               AS as_date,
        DATENAME(weekday, TRY_CONVERT(date, GMT)) AS day_name,
        MONTH(TRY_CONVERT(date, GMT))        AS month_no,
        DATENAME(month, TRY_CONVERT(date, GMT))   AS month_name,
        DATEPART(quarter, TRY_CONVERT(date, GMT)) AS qtr,
        EOMONTH(TRY_CONVERT(date, GMT))      AS month_end
    FROM week06.london_2014
""")

,raw_value,as_date,day_name,month_no,month_name,qtr,month_end
0,2014-1-1,2014-01-01,Wednesday,1,January,1,2014-01-31
1,2014-1-2,2014-01-02,Thursday,1,January,1,2014-01-31
2,2014-1-3,2014-01-03,Friday,1,January,1,2014-01-31
3,2014-1-4,2014-01-04,Saturday,1,January,1,2014-01-31
4,2014-1-5,2014-01-05,Sunday,1,January,1,2014-01-31


In [22]:
q("""
    SELECT DISTINCT TOP 5
        Period,
        DATEFROMPARTS(Period / 100, Period % 100, 1)               AS period_start,
        CONCAT(DATENAME(month, DATEFROMPARTS(Period / 100, Period % 100, 1)), ' ', Period / 100) AS period_label
    FROM week09.comtrade_milk_uk_monthly_14
    ORDER BY Period
""")

,Period,period_start,period_label
0,201401,2014-01-01,January 2014
1,201402,2014-02-01,February 2014
2,201403,2014-03-01,March 2014
3,201404,2014-04-01,April 2014
4,201405,2014-05-01,May 2014


In [24]:
q("""
    SELECT TOP 5
        country,
        [NY.GDP.MKTP.CD]                         AS gdp,
        ROUND([NY.GDP.MKTP.CD] / 1e9, 2)         AS gdp_bn,
        FLOOR(LOG10([NY.GDP.MKTP.CD]))           AS order_of_magnitude
    FROM week08.wb_gdp_2013
    WHERE [NY.GDP.MKTP.CD] > 0
    ORDER BY gdp DESC
""")


,country,gdp,gdp_bn,order_of_magnitude
0,World,"76,236,796,175,538.20","76,236.80",13.00
1,High income,"52,256,187,329,042.10","52,256.19",13.00
2,OECD members,"48,009,763,073,164.40","48,009.76",13.00
3,High income: OECD,"45,927,311,632,922.50","45,927.31",13.00
4,Low & middle income,"24,011,494,861,267.10","24,011.49",13.00


In [25]:
q("""
    SELECT
        CASE
            WHEN Applicant_Income <  3000 THEN '1. < 3k'
            WHEN Applicant_Income <  6000 THEN '2. 3k-6k'
            WHEN Applicant_Income < 10000 THEN '3. 6k-10k'
            WHEN Applicant_Income IS NULL THEN '9. unknown'
            ELSE '4. 10k+'
        END                                          AS income_band,
        COUNT(*)                                     AS applicants,
        ROUND(100.0 * AVG(1.0 * Loan_Approved), 1)   AS approval_rate_pct
    FROM week17.loan_approval_prediction
    GROUP BY
        CASE
            WHEN Applicant_Income <  3000 THEN '1. < 3k'
            WHEN Applicant_Income <  6000 THEN '2. 3k-6k'
            WHEN Applicant_Income < 10000 THEN '3. 6k-10k'
            WHEN Applicant_Income IS NULL THEN '9. unknown'
            ELSE '4. 10k+'
        END
    ORDER BY income_band
""")

,income_band,applicants,approval_rate_pct
0,4. 10k+,500,66.80


In [26]:
import numpy as np  
arr = np.array([1, 2, 3, 4, 5])  
print(arr)  
print(type(arr)) 

[1 2 3 4 5]
<class 'numpy.ndarray'>
